# Qwen2-VL-2B Fine-Tuning for AI Photography Critique
**Dataset:** `dulshanbandara/rpcd-elite-5k-v4` (3,504 samples)  
**Model:** `Qwen/Qwen2-VL-2B-Instruct`  
**Hardware:** Kaggle 2x T4 GPU (Free Tier)  
**Method:** LoRA + 4-bit quantization via Unsloth  
**Export:** GGUF Q4_K_M → Hugging Face Free CPU Space

## 1. Install Dependencies

In [ ]:
%%capture
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth[kaggle-new]",
    "unsloth_zoo",
    "bitsandbytes",
    "transformers==5.5.0",
    "trl>=0.11.0",
    "peft>=0.13.0",
    "accelerate",
    "qwen-vl-utils",
    "datasets",
    "huggingface_hub",
    "numpy<2.0.0",
    "gguf",
    "torch==2.10.0",
    "torchvision==0.25.0",
    "torchaudio==2.10.0",
    "-q",
], check=True)

# Clone llama.cpp for GGUF conversion later
if not os.path.exists("llama.cpp"):
    subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp",
                    "--depth", "1", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install",
        "-r", "llama.cpp/requirements.txt", "-q"
    ], check=True)

print("✅ All dependencies installed")

# Verify critical versions
import importlib.metadata
for pkg in ["transformers", "torch", "unsloth_zoo", "trl", "peft"]:
    try:
        print(f"  {pkg}: {importlib.metadata.version(pkg)}")
    except Exception:
        print(f"  {pkg}: NOT FOUND ⚠️")

print("🔄 Restarting kernel...")
os.kill(os.getpid(), 9)

## 2. Verify GPU Environment

In [ ]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

import torch

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Number of GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram = props.total_memory / 1024**3
    print(f'  GPU {i}: {props.name} — {vram:.1f} GB VRAM')

total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(torch.cuda.device_count())
) / 1024**3
print(f'\nTotal VRAM: {total_vram:.1f} GB')

## 3. Load Dataset from Kaggle

In [ ]:
import json
from pathlib import Path

# Search the entire Kaggle input directory for the dataset JSON
BASE_INPUT = Path('/kaggle/input/')
json_candidates = list(BASE_INPUT.rglob('*.json'))

print('JSON files found:', json_candidates)

if not json_candidates:
    raise FileNotFoundError(
        "No JSON files were found in /kaggle/input/. "
        "Please verify the dataset is attached in the right sidebar."
    )

# Use the first found JSON
JSON_PATH = json_candidates[0]
print(f"\nUsing JSON at: {JSON_PATH}")

# Set DATASET_ROOT to where we found the JSON
DATASET_ROOT = JSON_PATH.parent

with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

print(f'\nTotal samples: {len(raw_data)}')
print('Sample keys:', list(raw_data[0].keys()))
print('\nSample entry:')
print(json.dumps(raw_data[0], indent=2))

In [ ]:
# Locate the image directory within the dataset
image_dirs = [d for d in DATASET_ROOT.rglob('*') if d.is_dir()]
print('Directories found:')
for d in image_dirs:
    jpg_count = len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.jpg')))
    if jpg_count > 0:
        print(f'  {d}  ({jpg_count} images)')

In [ ]:
# Build a fast filename → path lookup table (O(n) instead of O(n*m) rglob per sample)
image_lookup = {}
for img_path in DATASET_ROOT.rglob('*'):
    if img_path.is_file() and img_path.suffix.lower() in ('.jpeg', '.jpg', '.png'):
        image_lookup[img_path.name] = img_path

print(f'Indexed {len(image_lookup)} images for fast lookup')

def remap_image_path(original_path: str) -> Path | None:
    """Convert Colab-style path to actual Kaggle path via filename lookup."""
    filename = Path(original_path).name
    return image_lookup.get(filename)

# Verify on first 3 samples
for item in raw_data[:3]:
    new_path = remap_image_path(item['original_image_path'])
    print(f"Original: {item['original_image_path']}")
    print(f"  Mapped: {new_path}")
    print(f"  Exists: {new_path.exists() if new_path else False}\n")

In [ ]:
# Build cleaned dataset with resolved image paths
valid_samples = []
missing = 0

for item in raw_data:
    resolved = remap_image_path(item['original_image_path'])
    if resolved and resolved.exists():
        valid_samples.append({
            **item,
            'image_path': str(resolved)
        })
    else:
        missing += 1

print(f'Valid samples: {len(valid_samples)}')
print(f'Missing images: {missing}')

## 4. Format Dataset into Qwen2-VL ChatML Conversations

In [ ]:
import json

SYSTEM_PROMPT = """You are a master photography instructor and aesthetic critic. Evaluate the provided image mechanically and artistically across composition, lighting, and technical execution. Provide scores (1-10) and detailed, actionable critiques for each dimension. Always respond in valid JSON format."""

USER_PROMPT = (
    "Analyse this photograph as an expert photography instructor. "
    "Evaluate composition, lighting, and technical execution. "
    "Provide an overall score, per-dimension scores and critiques, "
    "and a list of actionable improvement steps. "
    "Respond only with a JSON object."
)

def format_assistant_response(item: dict) -> str:
    """Format the structured critique as a JSON string."""
    response = {
        "overall_score": item["overall_score"],
        "composition": {
            "score": item["composition"]["score"],
            "critique": item["composition"]["critique"]
        },
        "lighting": {
            "score": item["lighting"]["score"],
            "critique": item["lighting"]["critique"]
        },
        "technical": {
            "score": item["technical"]["score"],
            "critique": item["technical"]["critique"]
        },
        "improvement_steps": item["improvement_steps"]
    }
    return json.dumps(response, indent=2)

print("✅ SYSTEM_PROMPT, USER_PROMPT, and format_assistant_response defined")

In [ ]:
from datasets import Dataset
import random

random.seed(42)
random.shuffle(valid_samples)

split = int(len(valid_samples) * 0.9)

def to_hf_dataset(samples):
    return Dataset.from_dict({
        "image_path":        [s["image_path"] for s in samples],
        "assistant_response": [format_assistant_response(s) for s in samples],
    })

train_dataset = to_hf_dataset(valid_samples[:split])
eval_dataset  = to_hf_dataset(valid_samples[split:])

print(f'Train: {len(train_dataset)} samples')
print(f'Eval:  {len(eval_dataset)} samples')
print("image_path type:",        type(train_dataset[0]["image_path"]))
print("assistant_response type:", type(train_dataset[0]["assistant_response"]))

## 5. Load Model with Unsloth (4-bit, LoRA)

In [ ]:
from unsloth import FastVisionModel

MODEL_ID = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)
print("✅ Unsloth imported and model loaded successfully!")

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0,
    bias        = "none",
    random_state= 42,
    use_rslora  = False,
)

model.print_trainable_parameters()
print('✅ LoRA adapters attached')

## 6. Data Collator for Vision-Language Training

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from PIL import Image

class PhotographyCollator:
    """
    Assembles Qwen2-VL conversations at collation time.
    Loads images from disk as PIL objects so Arrow never touches them,
    preventing the KeyError: 0 / empty-tensor issues.
    """
    def __init__(self, model, tokenizer):
        self._inner = UnslothVisionDataCollator(model, tokenizer)

    def __call__(self, batch):
        conversations = []
        for sample in batch:
            image = Image.open(sample["image_path"]).convert("RGB")
            conversations.append({
                "messages": [
                    {
                        "role": "system",
                        "content": [{"type": "text", "text": SYSTEM_PROMPT}]
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": USER_PROMPT}
                        ]
                    },
                    {
                        "role": "assistant",
                        "content": [{"type": "text", "text": sample["assistant_response"]}]
                    }
                ]
            })
        return self._inner(conversations)

data_collator = PhotographyCollator(model, tokenizer)
print("✅ Data collator ready")

### Sanity Check: Verify Collator Output

In [ ]:
# Verify collator produces valid batches before training
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    train_dataset,
    batch_size=1,
    collate_fn=data_collator,
)

batch = next(iter(debug_loader))

print("Keys in batch:", list(batch.keys()))
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:   ", batch["labels"].shape)
print("attention_mask: ", batch["attention_mask"].shape)

seq_len = batch["input_ids"].shape[1]
print(f"\nSequence length: {seq_len}  {'✅ OK' if seq_len > 0 else '❌ EMPTY — collator produced nothing'}")

# Single-sample collation test
try:
    one = data_collator([train_dataset[0]])
    print(f"\nSingle-sample collation: ✅ OK (shape: {one['input_ids'].shape})")
except Exception as e:
    print(f"\n❌ Collator error on single sample: {e}")

del debug_loader, batch
import gc; gc.collect()

## 7. Training Configuration

**Key fixes:**
- `average_tokens_across_devices=False` — prevents `'int' object has no attribute 'mean'` crash
- `logging_strategy/logging_first_step/logging_nan_inf_filter` — ensures loss values are printed during training

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

OUTPUT_DIR = "/kaggle/working/qwen2vl-photography-lora"

# ─── Custom callback to FORCE-PRINT loss every logging step ─────────
# Unsloth's patched training loop can suppress HF's built-in loss table.
# This callback bypasses that entirely.
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            loss = logs.get("loss", logs.get("train_loss"))
            eval_loss = logs.get("eval_loss")
            lr = logs.get("learning_rate")
            parts = [f"Step {state.global_step}/{state.max_steps}"]
            if loss is not None:
                parts.append(f"train_loss={loss:.4f}")
            if eval_loss is not None:
                parts.append(f"eval_loss={eval_loss:.4f}")
            if lr is not None:
                parts.append(f"lr={lr:.2e}")
            parts.append(f"epoch={state.epoch:.2f}")
            print(" | ".join(parts))
# ─────────────────────────────────────────────────────────────────────

sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 16,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 50,
    weight_decay                = 0.01,
    bf16                        = False,
    fp16                        = True,
    logging_strategy            = "steps",
    logging_steps               = 1,
    logging_first_step          = True,
    logging_nan_inf_filter      = False,
    eval_strategy               = "steps",
    eval_steps                  = 200,
    save_strategy               = "steps",
    save_steps                  = 200,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,
    ddp_find_unused_parameters  = False,
    gradient_checkpointing      = True,
    optim                       = "adamw_8bit",
    average_tokens_across_devices = False,
    dataset_text_field     = "",
    dataset_kwargs         = {"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model         = model,
    args          = sft_config,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    data_collator = data_collator,
)
trainer.add_callback(PrintLossCallback())
print("✅ SFTTrainer configured with loss-logging callback")

## 8. Train

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"Free VRAM: {(torch.cuda.mem_get_info()[0]/1024**3):.2f} GB")

In [ ]:
import time

print('🚀 Starting training...')
start = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - start
print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')
print(f'   Final train loss: {trainer_stats.training_loss:.4f}')

## 9. Save LoRA Adapter Locally

In [ ]:
from pathlib import Path

ADAPTER_SAVE_PATH = "/kaggle/working/qwen2vl-photography-adapter"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f'✅ LoRA adapter saved to {ADAPTER_SAVE_PATH}')

for f in Path(ADAPTER_SAVE_PATH).iterdir():
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name}  ({size_mb:.1f} MB)')

## 10. Push to Hugging Face Hub

In [ ]:
from huggingface_hub import login

# Add your HF token as a Kaggle secret:
# Kaggle → Add-ons → Secrets → Add secret named HF_TOKEN
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

login(token=hf_token)
print('✅ Logged in to Hugging Face')

In [ ]:
# ─── CHANGE THIS to your HF username ───────────────────────────────────────
HF_USERNAME  = "dulshanbandara"
HF_REPO_NAME = "qwen2vl-2b-photography-critique"
# ───────────────────────────────────────────────────────────────────────────

HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

model.push_to_hub(HF_REPO_ID, token=hf_token)
tokenizer.push_to_hub(HF_REPO_ID, token=hf_token)

print(f'✅ Adapter pushed to https://huggingface.co/{HF_REPO_ID}')

## 11. Quick Inference Test

In [ ]:
from unsloth import FastVisionModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor
from PIL import Image
import torch, json

# Switch to inference mode
FastVisionModel.for_inference(model)

# Pick a sample from eval set
sample = eval_dataset[0]
test_image = Image.open(sample["image_path"]).convert("RGB")

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {
                "type": "text",
                "text": (
                    "Analyse this photograph as an expert photography instructor. "
                    "Respond only with a JSON object."
                )
            }
        ]
    }
]

processor = AutoProcessor.from_pretrained(ADAPTER_SAVE_PATH)
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True
    )

generated = processor.batch_decode(
    output_ids[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)[0]

print("=== Model Output ===")
print(generated)

print("\n=== Ground Truth ===")
print(sample["assistant_response"])

In [ ]:
# Validate output is parseable JSON
try:
    parsed = json.loads(generated)
    print('✅ Output is valid JSON')
    print(f'   Overall score: {parsed.get("overall_score")}')
    print(f'   Composition score: {parsed.get("composition", {}).get("score")}')
    print(f'   Lighting score: {parsed.get("lighting", {}).get("score")}')
    print(f'   Technical score: {parsed.get("technical", {}).get("score")}')
except json.JSONDecodeError as e:
    print(f'⚠️  JSON parse error: {e}')
    print('   Try stripping markdown fences and retrying.')

## 12. Quantize to GGUF for Fast CPU Inference

Qwen2-VL has **two separate components** that must each be converted to GGUF:

| File | What it is | Format |
|------|-----------|--------|
| `qwen2vl-language-q4_k_m.gguf` | Qwen2 LLM text decoder | Q4_K_M (~1.3 GB) |
| `mmproj-model-f16.gguf` | Vision encoder + multimodal projector | F16 (~400 MB) |

Both files are required at inference time — llama-cpp-python loads them together via  
`Qwen2VLChatHandler(clip_model_path=mmproj)` + `Llama(model_path=llm)`.  

**Q4_K_M** is used for the LLM (best CPU speed/quality tradeoff).  
The mmproj stays **F16** — it is small (~400 MB) and quantizing it visibly degrades vision quality.

In [38]:
# Step 1: Merge LoRA adapters into float16 using PEFT (Workaround for Unsloth VLM merge bug)
import torch, gc
from transformers import Qwen2VLForConditionalGeneration
from peft import PeftModel

# Clear VRAM completely before merging
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

MERGED_SAVE_PATH = '/kaggle/working/qwen2vl-merged'

print("Loading base model in 16-bit...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu", # Merge on CPU to avoid OOM
)

print("Loading LoRA adapter...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)

print("Merging weights...")
merged_model = peft_model.merge_and_unload()

print("Saving merged model...")
merged_model.save_pretrained(MERGED_SAVE_PATH)
tokenizer.save_pretrained(MERGED_SAVE_PATH)

import subprocess
from pathlib import Path
result = subprocess.run(['du', '-sh', MERGED_SAVE_PATH], capture_output=True, text=True)
print(f'✅ Merged model saved to {MERGED_SAVE_PATH}')
print(f'   Disk usage: {result.stdout.split()[0]}')

print('\nCheckpoint files:')
for f in sorted(Path(MERGED_SAVE_PATH).iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024**2:.1f} MB)')

Loading base model in 16-bit...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loading LoRA adapter...
Merging weights...
Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved to /kaggle/working/qwen2vl-merged
   Disk usage: 4.2G

Checkpoint files:
  .cache  (0.0 MB)
  chat_template.jinja  (0.0 MB)
  config.json  (0.0 MB)
  generation_config.json  (0.0 MB)
  model.safetensors  (4213.4 MB)
  processor_config.json  (0.0 MB)
  tokenizer.json  (10.9 MB)
  tokenizer_config.json  (0.0 MB)


In [39]:
# Step 2: Build llama.cpp (CPU only for quantization tools)
import os, subprocess, sys
from pathlib import Path

if not os.path.exists('llama.cpp'):
    subprocess.run(['git', 'clone', 'https://github.com/ggerganov/llama.cpp',
                    '--depth', '1'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '-r', 'llama.cpp/requirements.txt'], check=True)

# Build without CUDA (GGML_CUDA=OFF). We only need this for 'llama-quantize'
# which runs perfectly fine (and fast) on CPU. This avoids Kaggle CUDA linking errors.
subprocess.run(
    'cd llama.cpp && cmake -B build '
    '-DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release && '
    'cmake --build build --config Release -j4',
    shell=True, check=True
)

print('✅ llama.cpp built')
for f in sorted(Path('llama.cpp/build/bin').iterdir()):
    print(f'  {f.name}')

CMAKE_BUILD_TYPE=Release


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.10.0
-- ggml commit:  72d693e-dirty
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-common
-- Configuring done (0.3s)
-- Generating done (0.4s)
-- Build files have been written to: /kaggle/working/llama.cpp/build
[  0%] Built target sha256
[  1%] Built target llama-common-base
[  1%] Built target cpp-httplib
[  1%] Built target sha1
[  2%] Built target xxhash
[  4%] Built target ggml-base
[  5%] Built target llama-llava-cli
[  6%] Built target llama-gemma3-cli
[  6%] Built target llama-minicpmv-cli
[  7%] Built target llama-qwen2vl-cli
[ 10%] Built target ggml-cpu
[ 11%] Built target ggml
[ 12%] Built target llama-gguf
[ 12%] Built target llama-gguf-hash
[ 42%] Buil

In [40]:
# Step 3a: Convert the LANGUAGE MODEL → F16 GGUF
import os, subprocess, sys, shutil, glob
from safetensors.torch import load_file, save_file

GGUF_DIR = '/kaggle/working/gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

# ─── WORKAROUND: Fix Safetensors for llama.cpp ───
# 1. Unsloth/PEFT merges vision weights as 'model.visual.*', but llama.cpp expects 'visual.*'.
# 2. PEFT accidentally creates rogue zero-bias tensors for vision projection layers.
# We fix both issues so llama.cpp's convert_hf_to_gguf.py can cleanly separate the LLM and mmproj.

CLEAN_MODEL_PATH = '/kaggle/working/qwen2vl-clean'
if os.path.exists(CLEAN_MODEL_PATH):
    shutil.rmtree(CLEAN_MODEL_PATH)
shutil.copytree(MERGED_SAVE_PATH, CLEAN_MODEL_PATH)

print("Fixing safetensor keys for llama.cpp...")
for st_file in glob.glob(f'{CLEAN_MODEL_PATH}/*.safetensors'):
    sd = load_file(st_file)
    sd_filtered = {}
    changed = False
    
    for k, v in sd.items():
        # Remove rogue bias tensors
        if "visual.blocks" in k and "attn.proj.bias" in k:
            changed = True
            continue
            
        # Rename model.visual.* to visual.*
        if k.startswith("model.visual."):
            new_k = k.replace("model.visual.", "visual.", 1)
            sd_filtered[new_k] = v
            changed = True
        else:
            sd_filtered[k] = v
            
    if changed:
        save_file(sd_filtered, st_file)

print("Converting LLM to F16 GGUF...")
# Without --mmproj, this converts ONLY the Language Model
subprocess.run([
    sys.executable, 'llama.cpp/convert_hf_to_gguf.py',
    CLEAN_MODEL_PATH,
    '--outtype', 'f16',
    '--outfile', f'{GGUF_DIR}/qwen2vl-language-f16.gguf'
], check=True)

lm_f16_size = os.path.getsize(f'{GGUF_DIR}/qwen2vl-language-f16.gguf') / 1024**3
print(f'\n✅ Language model F16 GGUF: {lm_f16_size:.2f} GB')

Fixing safetensor keys for llama.cpp...
Converting LLM to F16 GGUF...


INFO:hf-to-gguf:Loading model: qwen2vl-clean
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Qwen2VLForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {15


✅ Language model F16 GGUF: 2.88 GB


In [41]:
# Step 3b: Convert the VISION ENCODER (mmproj) → F16 GGUF
import subprocess, os, glob

# In recent llama.cpp versions, convert_image_encoder_to_gguf.py was deleted.
# Instead, we use convert_hf_to_gguf.py with the --mmproj flag!

subprocess.run([
    sys.executable, 'llama.cpp/convert_hf_to_gguf.py',
    CLEAN_MODEL_PATH,
    '--mmproj',
    '--outtype', 'f16',
    '--outfile', f'{GGUF_DIR}/qwen2vl-mmproj-f16.gguf'
], check=True)

mmproj_path = f'{GGUF_DIR}/qwen2vl-mmproj-f16.gguf'
mmproj_mb = os.path.getsize(mmproj_path) / 1024**2
print(f'✅ Vision encoder (mmproj) F16 GGUF: {mmproj_mb:.0f} MB')

INFO:hf-to-gguf:Loading model: qwen2vl-clean
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Qwen2VLForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:v.blk.0.attn_out.weight,         torch.float16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.attn_q.bias,             torch.float16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_k.bias,             torch.float16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_v.bias,             torch.float16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_q.weight,           torch.float16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.attn_k.weight,           torch.float16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.attn_v.weight,           torch.float16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.ffn_up.bias,

✅ Vision encoder (mmproj) F16 GGUF: 1270 MB


In [42]:
# Step 4: Quantize the LLM F16 → Q4_K_M
import subprocess

subprocess.run([
    'llama.cpp/build/bin/llama-quantize',
    f'{GGUF_DIR}/qwen2vl-language-f16.gguf',
    f'{GGUF_DIR}/qwen2vl-language-q4_k_m.gguf',
    'Q4_K_M'
], check=True)

q4_size   = os.path.getsize(f'{GGUF_DIR}/qwen2vl-language-q4_k_m.gguf') / 1024**3
mmproj_gb = os.path.getsize(mmproj_path) / 1024**3

print('\n── Deployment size summary ──')
print(f'  LLM F16 (discarded):   {lm_f16_size:.2f} GB')
print(f'  LLM Q4_K_M (deploy):   {q4_size:.2f} GB')
print(f'  mmproj F16 (deploy):   {mmproj_gb:.2f} GB')
print(f'  Total on HF Space:     {q4_size + mmproj_gb:.2f} GB')

llama_print_build_info: build = 1 (72d693e)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/kaggle/working/gguf/qwen2vl-language-f16.gguf' to '/kaggle/working/gguf/qwen2vl-language-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 27 key-value pairs and 338 tensors from /kaggle/working/gguf/qwen2vl-language-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2vl
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 1
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.001000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.010000
llama_model_lo


main: quantize time = 76666.18 ms
main:    total time = 76666.18 ms

── Deployment size summary ──
  LLM F16 (discarded):   2.88 GB
  LLM Q4_K_M (deploy):   0.92 GB
  mmproj F16 (deploy):   1.24 GB
  Total on HF Space:     2.16 GB


### Push Both GGUF Files to Hugging Face Hub

In [43]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)

HF_USERNAME  = 'dulshanbandara'
HF_REPO_NAME = 'qwen2vl-2b-photography-critique'
HF_GGUF_REPO = f'{HF_USERNAME}/{HF_REPO_NAME}-GGUF'

api.create_repo(HF_GGUF_REPO, repo_type='model', exist_ok=True)

# Upload LLM Q4_K_M
print('Uploading language model Q4_K_M...')
api.upload_file(
    path_or_fileobj=f'{GGUF_DIR}/qwen2vl-language-q4_k_m.gguf',
    path_in_repo='qwen2vl-language-q4_k_m.gguf',
    repo_id=HF_GGUF_REPO,
    repo_type='model',
)
print('✅ LLM Q4_K_M pushed')

# Upload mmproj F16
print('Uploading vision encoder (mmproj)...')
api.upload_file(
    path_or_fileobj=mmproj_path,
    path_in_repo='qwen2vl-mmproj-f16.gguf',
    repo_id=HF_GGUF_REPO,
    repo_type='model',
)
print('✅ mmproj F16 pushed')

# Upload tokenizer/processor
tokenizer.push_to_hub(HF_GGUF_REPO, token=hf_token)
print('✅ Tokenizer pushed')

print(f'\n🎉 Repo: https://huggingface.co/{HF_GGUF_REPO}')
print('\n── Load in your HF CPU Space ──')
print('from llama_cpp import Llama')
print('from llama_cpp.llama_chat_format import Qwen2VLChatHandler')
print('from huggingface_hub import hf_hub_download')
print('chat_handler = Qwen2VLChatHandler(')
print('    clip_model_path=hf_hub_download("dulshanbandara/qwen2vl-2b-photography-critique-GGUF", "qwen2vl-mmproj-f16.gguf"),')
print(')')
print('llm = Llama(')
print('    model_path=hf_hub_download("dulshanbandara/qwen2vl-2b-photography-critique-GGUF", "qwen2vl-language-q4_k_m.gguf"),')
print('    chat_handler=chat_handler, n_ctx=4096, n_gpu_layers=0')
print(')')

Uploading language model Q4_K_M...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ LLM Q4_K_M pushed
Uploading vision encoder (mmproj)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ mmproj F16 pushed


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Tokenizer pushed

🎉 Repo: https://huggingface.co/dulshanbandara/qwen2vl-2b-photography-critique-GGUF

── Load in your HF CPU Space ──
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Qwen2VLChatHandler
from huggingface_hub import hf_hub_download
chat_handler = Qwen2VLChatHandler(
    clip_model_path=hf_hub_download("dulshanbandara/qwen2vl-2b-photography-critique-GGUF", "qwen2vl-mmproj-f16.gguf"),
)
llm = Llama(
    model_path=hf_hub_download("dulshanbandara/qwen2vl-2b-photography-critique-GGUF", "qwen2vl-language-q4_k_m.gguf"),
    chat_handler=chat_handler, n_ctx=4096, n_gpu_layers=0
)


### CPU Inference Test — Vision + Language Together
Simulates exactly how the model runs on the HF Free CPU Space (`n_gpu_layers=0`).

In [45]:
# Install llama-cpp-python with CUDA for faster testing on Kaggle.
import subprocess, sys, os
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'llama-cpp-python', '-q'],
    env={**os.environ, 'CMAKE_ARGS': '-DGGML_CUDA=on'},
    check=True
)

from llama_cpp import Llama
from llama_cpp.llama_chat_format import Qwen2VLChatHandler
import base64, json, time

# Load BOTH parts — required for Qwen2-VL
chat_handler = Qwen2VLChatHandler(
    clip_model_path=mmproj_path,
    verbose=False,
)
llm = Llama(
    model_path=f'{GGUF_DIR}/qwen2vl-language-q4_k_m.gguf',
    chat_handler=chat_handler,
    n_ctx=4096,
    n_gpu_layers=0,
    verbose=False,
)

# Load test image from eval set
sample = eval_dataset[0]
test_img_path = sample["image_path"]
with open(test_img_path, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()
img_uri = f'data:image/jpeg;base64,{img_b64}'

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": img_uri}},
            {"type": "text", "text": "Analyse this photograph. Respond only with a JSON object."},
        ]
    }
]

t0 = time.time()
response = llm.create_chat_completion(
    messages=messages,
    max_tokens=512,
    temperature=0.1,
)
elapsed = time.time() - t0

result_text = response["choices"][0]["message"]["content"].strip()
print(f'⏱  CPU inference time: {elapsed:.1f}s')
print("\n=== Model Output ===")
print(result_text)

try:
    parsed = json.loads(result_text)
    print("\n✅ Valid JSON")
    print(f'   overall_score:     {parsed.get("overall_score")}')
    print(f'   composition score: {parsed.get("composition",{}).get("score")}')
    print(f'   lighting score:    {parsed.get("lighting",{}).get("score")}')
    print(f'   technical score:   {parsed.get("technical",{}).get("score")}')
except json.JSONDecodeError:
    print("\n⚠️  JSON parse error — model may need more fine-tuning epochs")

  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'llama-cpp-python', '-q']' returned non-zero exit status 1.